In [51]:
# Import Python packages 
import pandas as pd
import cassandra
import re
import os
import glob
import numpy as np
import json
import csv
from prettytable import PrettyTable

In [52]:
# Read CSV files and combine them in one CSV file

folder_path = os.path.join(os.getcwd(), 'CSVs_data')
files = glob.glob(os.path.join(folder_path, '*.csv'))

with open('Result.csv', 'w', encoding='utf8', newline='') as out_file:
    writer = csv.writer(out_file, quoting=csv.QUOTE_ALL)
    
    writer.writerow([
        'artist','firstName','gender','itemInSession','lastName',
        'length','level','location','sessionId','song','userId'
    ])
    
    for file in files:
        with open(file, 'r', encoding='utf8') as f:
            reader = csv.reader(f)
            next(reader)  # skip header
            
            for row in reader:
                if row[0] == '':
                    continue
                
                writer.writerow([
                    row[0], row[2], row[3], row[4], row[5],
                    row[6], row[7], row[8], row[12], row[13], row[16]
                ])

In [53]:
# connect to Cassandra 

from cassandra.cluster import Cluster
cluster = Cluster()

session = cluster.connect()

In [55]:
try:
    session.execute("""
    CREATE KEYSPACE IF NOT EXISTS Event_K 
    WITH REPLICATION = 
    { 'class' : 'SimpleStrategy', 'replication_factor' : 1 }"""
)
    session.set_keyspace('Event_K')
except Exception as e:
    print(e)
    

In [31]:
# we need to do the following queries. with Apache Cassandra you model the database tables on the queries you want to run.

# queries to ask the following three questions of the data:

# 1. Give me the artist, song title and song's length in the music app history that was heard during sessionId = 338, 
#    and itemInSession = 4

# 2. Give me only the following: name of artist, song (sorted by itemInSession) and user (first and last name) for userid = 10,
#    sessionid = 182

# 3. Give me every user name (first and last) in my music app history who listened to the song 'All Hands Against His Own'

In [56]:
# Query 1:  Give me the artist, song title and song's length in the music app history that was heard during \
# sessionId = 338, and itemInSession = 4

query = "CREATE TABLE IF NOT EXISTS song_info_by_session "
query = query + "(sessionId int, itemInSession int, artist text, \
                  song_title text, length float, \
                  PRIMARY KEY (sessionId, itemInSession))"
try:
    session.execute(query)
except Exception as e:
    print(e)  
    
query = "INSERT INTO song_info_by_session (sessionId, itemInSession, artist, \
                  song_title, length)"
query = query + " VALUES (%s, %s, %s, %s, %s)"

with open('Result.csv', encoding = 'utf8') as f:
    csvreader = csv.reader(f)
    next(csvreader) # skip header
    for line in csvreader:
        length = float(line[5])
        sessionId = int(line[8])
        itemInSession = int(line[3])
        session.execute(query, (sessionId, itemInSession, line[0], line[9], length))

In [57]:
## verifying the data was entered into the table
query = "SELECT artist, song_title, length from song_info_by_session WHERE sessionId = 338 and itemInSession = 4"
try:
    rows = session.execute(query)
except Exception as e:
    print(e)
    
x = PrettyTable()
x.field_names = ["artist", "song title", "length"]
for row in rows:
    x.add_row([row.artist, row.song_title, row.length])
print(x)

+-----------+---------------------------------+--------------------+
|   artist  |            song title           |       length       |
+-----------+---------------------------------+--------------------+
| Faithless | Music Matters (Mark Knight Dub) | 495.30731201171875 |
+-----------+---------------------------------+--------------------+


In [58]:
## Query 2: Give me only the following: name of artist, song (sorted by itemInSession) and user (first and last name)\
## for userid = 10, sessionid = 182
## adding artist column as a CLUSTERING COLUMN to get song with the same title for different artists

query = "CREATE TABLE IF NOT EXISTS song_info_by_itemInSession "
query = query + "(userId int, sessionId int, itemInSession int, artist text, song_title text, \
                first_name text, last_name text,  \
                  PRIMARY KEY ((userId, sessionId), itemInSession, first_name, last_name))"
try:
    session.execute(query)
except Exception as e:
    print(e)
            

query = "INSERT INTO song_info_by_itemInSession (userId, sessionId,  itemInSession, artist, song_title,\
                first_name, last_name)"
query = query + " VALUES (%s, %s, %s, %s, %s, %s, %s)"

with open('Result.csv', encoding = 'utf8') as f:
    csvreader = csv.reader(f)
    next(csvreader) # skip header
    for line in csvreader:
        userId = int(line[10])
        sessionId = int(line[8])
        itemInSession = int(line[3])
        session.execute(query, (userId, sessionId, itemInSession, line[0], line[9], line[1], line[4]))

In [60]:
## verifying the data was entered into the table
query = "SELECT artist, song_title from song_info_by_itemInSession WHERE userId = 10 AND sessionId = 182"
try:
    rows = session.execute(query)
except Exception as e:
    print(e)
x = PrettyTable()
x.field_names = ["artist", "song title"]
for row in rows:
    x.add_row([row.artist, row.song_title])
print(x)

+-------------------+------------------------------------------------------+
|       artist      |                      song title                      |
+-------------------+------------------------------------------------------+
|  Down To The Bone |                  Keep On Keepin' On                  |
|    Three Drives   |                     Greece 2000                      |
| Sebastien Tellier |                      Kilometer                       |
|   Lonnie Gordon   | Catch You Baby (Steve Pitron & Max Sanna Radio Edit) |
+-------------------+------------------------------------------------------+


In [61]:
## Query 3: Give me every user name (first and last) in my music app history who listened to the song 'All Hands Against His Own'

query = "CREATE TABLE IF NOT EXISTS user_data_listened_to_a_song "
query = query + "(song_title text, userId int, first_name text, last_name text, \
                  PRIMARY KEY (song_title, userId))"
try:
    session.execute(query)
except Exception as e:
    print(e)

    
    
query = "INSERT INTO user_data_listened_to_a_song (song_title, userId, first_name, last_name )"
query = query + " VALUES (%s, %s, %s, %s)"

with open('Result.csv', encoding = 'utf8') as f:
    csvreader = csv.reader(f)
    next(csvreader) # skip header
    for line in csvreader:
        userId = int(line[10])
        session.execute(query, (line[9], userId, line[1], line[4]))

In [62]:
## verifying the data was entered into the table
query = "SELECT first_name, last_name from user_data_listened_to_a_song WHERE song_title = 'All Hands Against His Own'"
try:
    rows = session.execute(query)
except Exception as e:
    print(e)
x = PrettyTable()
x.field_names = ["first name", "last name "]
for row in rows:
    x.add_row([row.first_name, row.last_name])
print(x)

+------------+------------+
| first name | last name  |
+------------+------------+
| Jacqueline |   Lynch    |
|   Tegan    |   Levine   |
|    Sara    |  Johnson   |
+------------+------------+


In [63]:
query = "drop table song_info_by_session"
try:
    rows = session.execute(query)
except Exception as e:
    print(e)
    
query = "drop table song_info_by_itemInSession"
try:
    rows = session.execute(query)
except Exception as e:
    print(e)
    
query = "drop table user_data_listened_to_a_song"
try:
    rows = session.execute(query)
except Exception as e:
    print(e)

In [64]:
session.shutdown()
cluster.shutdown()